# Step 2 — Full training + Kaggle submission\nconvnextv2_base | efficientnetv2_m | deit3_base + mpnet embeddings\nepochs fixes depuis best_epochs.json (step 1)

In [ ]:
!pip install timm==1.0.3 albumentations sentence-transformers -q
print('OK')

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.6/43.6 kB 3.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.3/2.3 MB 105.3 MB/s eta 0:00:00
OK


In [ ]:
import os, gc, math, random, time, json, warnings, sys
from pathlib import Path
import cv2
import numpy as np
import pandas as pd
from sklearn.metrics import mean_absolute_error
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torch.amp import autocast, GradScaler
import albumentations as A
from albumentations.pytorch import ToTensorV2
import timm
warnings.filterwarnings('ignore')
print('PyTorch:', torch.__version__, '| GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU')

PyTorch: 2.11.0+cu128 | GPU: NVIDIA A100-SXM4-40GB


In [ ]:
IS_KAGGLE = os.path.exists('/kaggle/input')
IS_COLAB  = 'google.colab' in sys.modules

DEBUG        = False
DEBUG_EPOCHS = 1

if IS_COLAB:
    from google.colab import drive, files
    drive.mount('/content/drive')
    WORK_STEP1  = Path('/content/drive/MyDrive/food_step1')
    WORK        = Path('/content/drive/MyDrive/food_step2') if not DEBUG else Path('/content/food_step2_debug')
    KAGGLE_DATA = Path('/content/data')
    KAGGLE_DATA.mkdir(parents=True, exist_ok=True)
    if not (KAGGLE_DATA / 'train_labels.csv').exists():
        uploaded = files.upload()
        !mkdir -p ~/.kaggle
        !mv kaggle.json ~/.kaggle/
        !chmod 600 ~/.kaggle/kaggle.json
        !pip install kaggle -q
        !kaggle competitions download -c m2-food-calorie-estimation -p /content/data
        !unzip -q /content/data/m2-food-calorie-estimation.zip -d /content/data/
    DESC_CSV  = Path('/content/drive/MyDrive/food_descriptions/descriptions_merged.csv')
    EMB_CACHE = Path('/content/drive/MyDrive/food_descriptions/embeddings_mpnet.npz')
elif IS_KAGGLE:
    WORK_STEP1 = None
    for candidate in [
        Path('/kaggle/input/food-step1-results'),
        Path('/kaggle/input/datasets/yacineabdelouhab/food-step1-results'),
    ]:
        if (candidate / 'best_epochs.json').exists():
            WORK_STEP1 = candidate; break
    WORK = Path('/kaggle/working')
    for candidate in [
        Path('/kaggle/input/m2-food-calorie-estimation'),
        Path('/kaggle/input/competitions/m2-food-calorie-estimation'),
    ]:
        if (candidate / 'train_labels.csv').exists():
            KAGGLE_DATA = candidate; break
    else:
        KAGGLE_DATA = list(Path('/kaggle/input').rglob('train_labels.csv'))[0].parent
    DESC_CSV  = Path('/kaggle/input/datasets/yacineabdelouhab/food-descriptions-moondream/descriptions_merged.csv')
    EMB_CACHE = Path('/kaggle/working/embeddings_mpnet.npz')
else:
    WORK_STEP1 = Path('.')
    WORK       = Path('.')
    KAGGLE_DATA = Path(r'C:\Users\Abdel\.cache\kagglehub\competitions\m2-food-calorie-estimation')
    DESC_CSV  = Path(r'C:\Users\Abdel\Desktop\M2Dauphine\DL_Image\Projet\description_textuel\descriptions_merged.csv')
    EMB_CACHE = WORK / 'embeddings_mpnet.npz'

WORK.mkdir(parents=True, exist_ok=True)
os.chdir(WORK)
CKPT_DIR = WORK / 'checkpoints'; CKPT_DIR.mkdir(exist_ok=True)
OUT_DIR  = WORK / 'outputs';     OUT_DIR.mkdir(exist_ok=True)

SEED=42; PRED_MIN=30.0; PRED_MAX=3600.0
NUM_WORKERS = 2 if IS_KAGGLE else 4

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(device, '|', 'debug' if DEBUG else 'normal', '|', WORK)
print('EMB_CACHE:', EMB_CACHE)

Mounted at /content/drive


Saving kaggle.json to kaggle.json
100% 2.06G/2.06G [01:43<00:00, 21.5MB/s]

cuda | normal | /content/drive/MyDrive/food_step2
EMB_CACHE: /content/drive/MyDrive/food_descriptions/embeddings_mpnet.npz


In [ ]:
EPOCH_SCALE = 1.3
DEFAULT_EPOCHS = {'convnextv2_base_mm': 50, 'efficientnetv2_m_mm': 50, 'deit3_base_mm': 50, 'convnextv2_base_augmax': 50}

best_epochs_path = (WORK_STEP1 / 'best_epochs.json') if WORK_STEP1 else None
nm_weights_path  = (WORK_STEP1 / 'nelder_mead_weights.json') if WORK_STEP1 else None

if DEBUG:
    BEST_EPOCHS = {k: DEBUG_EPOCHS for k in DEFAULT_EPOCHS}
elif best_epochs_path and best_epochs_path.exists():
    raw = json.loads(best_epochs_path.read_text())
    BEST_EPOCHS = {k: max(1, int(round(v * EPOCH_SCALE))) for k, v in raw.items()}
else:
    BEST_EPOCHS = DEFAULT_EPOCHS

NM_WEIGHTS = None
if not DEBUG and nm_weights_path and nm_weights_path.exists():
    NM_WEIGHTS = json.loads(nm_weights_path.read_text())

print('epochs (x1.3):', BEST_EPOCHS)
print('nm_weights:', NM_WEIGHTS)

epochs (x1.3): {'convnextv2_base_mm': 40, 'efficientnetv2_m_mm': 84, 'deit3_base_mm': 42, 'convnextv2_base_augmax': 79}
nm_weights: {'convnextv2_base_augmax': 0.8265267207744856, 'convnextv2_base_mm': 0.06792383161571705, 'deit3_base_mm': 0.10554944740203964, 'efficientnetv2_m_mm': 2.0775755127171622e-10}


In [ ]:
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
torch.backends.cudnn.benchmark = True

TRAIN_IMGS = KAGGLE_DATA / 'train' / 'images'
TEST_IMGS  = KAGGLE_DATA / 'test'  / 'images'

def resolve_image_path(img_dir, filename):
    p = img_dir / filename
    if p.exists(): return str(p)
    alt = p.with_suffix('.png' if p.suffix.lower()=='.jpg' else '.jpg')
    return str(alt) if alt.exists() else str(p)

train_df = pd.read_csv(KAGGLE_DATA / 'train_labels.csv')
test_df  = pd.read_csv(KAGGLE_DATA / 'test_ids.csv')
train_df['path'] = train_df['filename'].apply(lambda f: resolve_image_path(TRAIN_IMGS, f))
test_df['path']  = test_df['filename'].apply(lambda f: resolve_image_path(TEST_IMGS, f))
train_df['target_log'] = np.log1p(train_df['calories'].astype(float))

df_all_train = train_df.reset_index(drop=True)
print(f'train={len(df_all_train)} kaggle={len(test_df)}')

train=3098 kaggle=547


In [ ]:
from sentence_transformers import SentenceTransformer

df_desc = pd.read_csv(DESC_CSV)
descriptions = dict(zip(df_desc['image_id'].astype(str), df_desc['description'].fillna('food dish')))

if EMB_CACHE.exists():
    tmp = np.load(EMB_CACHE, allow_pickle=True)
    if tmp['vecs'].shape[1] != 768:
        EMB_CACHE.unlink()

if EMB_CACHE.exists():
    saved  = np.load(EMB_CACHE, allow_pickle=True)
    id2emb = {k: saved['vecs'][i] for i, k in enumerate(saved['ids'])}
else:
    st_model = SentenceTransformer('all-mpnet-base-v2', device='cpu')
    ids_list = list(descriptions.keys())
    vecs     = st_model.encode([descriptions[k] for k in ids_list], batch_size=32,
                                show_progress_bar=True, normalize_embeddings=True).astype(np.float32)
    np.savez(EMB_CACHE, ids=np.array(ids_list), vecs=vecs)
    id2emb = {k: vecs[i] for i, k in enumerate(ids_list)}
    del st_model; gc.collect()

TEXT_DIM = next(iter(id2emb.values())).shape[0]
print(f'{len(id2emb)} embeddings x {TEXT_DIM}-dim')

3645 embeddings x 768-dim


In [ ]:
class MultimodalFoodDataset(Dataset):
    def __init__(self, df, transforms, id2emb, is_train=True):
        self.df         = df.reset_index(drop=True)
        self.transforms = transforms
        self.id2emb     = id2emb
        self.is_train   = is_train
        self.fallback   = np.zeros(TEXT_DIM, dtype=np.float32)

    def __len__(self): return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        img = cv2.imread(row.path, cv2.IMREAD_COLOR)
        if img is None: raise FileNotFoundError(row.path)
        img      = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
        img      = self.transforms(image=img)['image']
        text_emb = torch.tensor(self.id2emb.get(str(row.image_id), self.fallback), dtype=torch.float32)
        if self.is_train:
            return img, text_emb, torch.tensor(float(row.target_log), dtype=torch.float32)
        return img, text_emb, str(row.image_id)

def make_loader(df, transforms, batch_size, is_train, shuffle, drop_last=False):
    ds = MultimodalFoodDataset(df, transforms, id2emb, is_train=is_train)
    return DataLoader(ds, batch_size=batch_size, shuffle=shuffle,
                      num_workers=NUM_WORKERS, pin_memory=True, drop_last=drop_last)

In [ ]:
BACKBONES_WITH_IMG_SIZE = ('swin', 'vit', 'deit', 'beit')

class MultimodalRegressor(nn.Module):
    def __init__(self, backbone_name, img_size, text_dim, dropout=0.25):
        super().__init__()
        needs_img_size = any(k in backbone_name for k in BACKBONES_WITH_IMG_SIZE)
        extra = {'img_size': img_size} if needs_img_size else {}
        self.backbone = timm.create_model(
            backbone_name, pretrained=True, num_classes=0, global_pool='avg', **extra)
        fused_dim = self.backbone.num_features + text_dim
        self.head = nn.Sequential(
            nn.LayerNorm(fused_dim),
            nn.Linear(fused_dim, 512), nn.GELU(), nn.Dropout(dropout),
            nn.Linear(512, 128),       nn.GELU(), nn.Dropout(dropout / 2),
            nn.Linear(128, 1),
        )
        print(f'  fused={fused_dim}')

    def forward(self, img, text_emb):
        return self.head(torch.cat([self.backbone(img), text_emb], dim=1)).squeeze(1)

def get_optimizer_llrd(model, base_lr, wd, n_groups=4, decay=0.25):
    bp = [(n,p) for n,p in model.backbone.named_parameters() if p.requires_grad]
    hp = [(n,p) for n,p in model.head.named_parameters()    if p.requires_grad]
    gs = max(1, len(bp)//n_groups); pg = []
    for g in range(n_groups):
        s = g*gs; e = (g+1)*gs if g < n_groups-1 else len(bp)
        lr_g = base_lr * (decay**(n_groups-1-g)); chunk = bp[s:e]
        wd_  = [p for n,p in chunk if p.ndim>1 and not n.endswith('bias') and 'norm' not in n.lower()]
        nwd  = [p for n,p in chunk if p.ndim<=1 or n.endswith('bias') or 'norm' in n.lower()]
        if wd_:  pg.append({'params':wd_,  'lr':lr_g, 'weight_decay':wd})
        if nwd:  pg.append({'params':nwd,  'lr':lr_g, 'weight_decay':0.})
    hwd  = [p for n,p in hp if p.ndim>1 and not n.endswith('bias') and 'norm' not in n.lower()]
    hnwd = [p for n,p in hp if p.ndim<=1 or n.endswith('bias') or 'norm' in n.lower()]
    if hwd:  pg.append({'params':hwd,  'lr':base_lr, 'weight_decay':wd})
    if hnwd: pg.append({'params':hnwd, 'lr':base_lr, 'weight_decay':0.})
    return torch.optim.AdamW(pg)

In [ ]:
def preds_to_calories(p): return np.clip(np.expm1(p), PRED_MIN, PRED_MAX)

MIXUP_ALPHA = 0.4; AUG_MIX_PROB = 0.3

def mixup(imgs, texts, targets):
    lam = np.random.beta(MIXUP_ALPHA, MIXUP_ALPHA)
    idx = torch.randperm(imgs.size(0), device=imgs.device)
    return lam*imgs+(1-lam)*imgs[idx], lam*texts+(1-lam)*texts[idx], lam*targets+(1-lam)*targets[idx]

def train_one_epoch(model, loader, optimizer, scheduler, scaler, criterion):
    model.train(); total = 0.; optimizer.zero_grad(set_to_none=True)
    for imgs, texts, targets in loader:
        imgs    = imgs.to(device, non_blocking=True)
        texts   = texts.to(device, non_blocking=True)
        targets = targets.to(device, non_blocking=True)
        if random.random() < AUG_MIX_PROB:
            imgs, texts, targets = mixup(imgs, texts, targets)
        with autocast(device_type='cuda', enabled=torch.cuda.is_available()):
            loss = criterion(model(imgs, texts), targets)
        scaler.scale(loss).backward()
        scaler.unscale_(optimizer); nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        scaler.step(optimizer); scaler.update(); optimizer.zero_grad(set_to_none=True)
        if scheduler: scheduler.step()
        total += float(loss.item())
    return total / len(loader)

@torch.no_grad()
def predict(model, loader):
    model.eval(); all_ids, all_preds = [], []
    for imgs, texts, ids in loader:
        imgs  = imgs.to(device, non_blocking=True)
        texts = texts.to(device, non_blocking=True)
        with autocast(device_type='cuda', enabled=torch.cuda.is_available()):
            out      = model(imgs, texts)
            out_flip = model(torch.flip(imgs, dims=[3]), texts)
        all_preds.append(((out+out_flip)/2).float().cpu().numpy())
        all_ids.extend(ids if isinstance(ids, list) else list(ids))
    return all_ids, np.concatenate(all_preds)

In [ ]:
MODELS = [
    ('convnextv2_base.fcmae_ft_in22k_in1k',        224, 'convnextv2_base_mm',  64),
    ('tf_efficientnetv2_m.in21k_ft_in1k',           224, 'efficientnetv2_m_mm', 64),
    ('deit3_base_patch16_224.fb_in22k_ft_in1k',     224, 'deit3_base_mm',       64),
]

LR=2e-4; WD=1e-2; DROPOUT=0.25

# Models included in final blend (efficientnet excluded — NM weight was 0 in step 1)
BLEND_MODELS = ['convnextv2_base_mm', 'deit3_base_mm', 'convnextv2_base_augmax']
# Fallback weights if NM_WEIGHTS not available or key missing
# augmax capped lower: generalized worse in step1 (44.45→48.98 on Kaggle)
BLEND_FALLBACK_W = {'convnextv2_base_mm': 0.55, 'deit3_base_mm': 0.35, 'convnextv2_base_augmax': 0.10}

for _, _, name, _ in MODELS:
    print(f'  {name}: {BEST_EPOCHS.get(name, 50)} epochs')
print(f'  convnextv2_base_augmax: {BEST_EPOCHS.get("convnextv2_base_augmax", 50)} epochs')

  convnextv2_base_mm: 40 epochs
  efficientnetv2_m_mm: 84 epochs
  deit3_base_mm: 42 epochs
  convnextv2_base_augmax: 79 epochs


In [ ]:
all_kaggle_preds = {}

for backbone_name, img_size, exp_name, bs in MODELS:
    print(f'\n{exp_name}')
    EPOCHS     = BEST_EPOCHS.get(exp_name, 50)
    ckpt_path  = CKPT_DIR / f'{exp_name}_full.pth'
    preds_path = OUT_DIR  / f'{exp_name}_full_preds.npz'

    if not DEBUG and ckpt_path.exists() and preds_path.exists():
        saved = np.load(preds_path)
        all_kaggle_preds[exp_name] = {'ids': list(saved['kaggle_ids']), 'preds': saved['kaggle_preds']}
        print(f'  skip | {len(saved["kaggle_ids"])} preds')
        continue

    train_tfm = A.Compose([
        A.RandomResizedCrop(size=(img_size,img_size), scale=(0.65,1.0), ratio=(0.80,1.20), interpolation=cv2.INTER_AREA),
        A.HorizontalFlip(p=0.5),
        A.ColorJitter(brightness=0.25, contrast=0.25, saturation=0.25, hue=0.08, p=0.6),
        A.Normalize(mean=(0.485,0.456,0.406), std=(0.229,0.224,0.225)),
        ToTensorV2(),
    ])
    test_tfm = A.Compose([
        A.LongestMaxSize(max_size=img_size),
        A.PadIfNeeded(min_height=img_size, min_width=img_size, border_mode=cv2.BORDER_REFLECT_101),
        A.CenterCrop(height=img_size, width=img_size),
        A.Normalize(mean=(0.485,0.456,0.406), std=(0.229,0.224,0.225)),
        ToTensorV2(),
    ])

    train_loader  = make_loader(df_all_train, train_tfm, bs, is_train=True,  shuffle=True, drop_last=True)
    kaggle_loader = make_loader(test_df,      test_tfm,  bs, is_train=False, shuffle=False)

    model     = MultimodalRegressor(backbone_name, img_size, TEXT_DIM, DROPOUT).to(device)
    optimizer = get_optimizer_llrd(model, base_lr=LR, wd=WD)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS*len(train_loader), eta_min=1e-7)
    scaler    = GradScaler(enabled=torch.cuda.is_available())
    criterion = nn.HuberLoss(delta=0.5)

    best_loss = float('inf')
    for epoch in range(1, EPOCHS+1):
        t0         = time.time()
        train_loss = train_one_epoch(model, train_loader, optimizer, scheduler, scaler, criterion)
        if train_loss < best_loss:
            best_loss = train_loss
            torch.save({'model_state': model.state_dict(), 'epoch': epoch,
                        'backbone': backbone_name, 'img_size': img_size}, ckpt_path)
        print(f'  {epoch:03d}/{EPOCHS} | train={train_loss:.4f} best={best_loss:.4f} | {time.time()-t0:.0f}s', flush=True)

    ckpt = torch.load(ckpt_path, map_location=device, weights_only=False)
    model.load_state_dict(ckpt['model_state'])

    kids, kpl = predict(model, kaggle_loader)
    kpc = preds_to_calories(kpl)
    np.savez(preds_path, kaggle_ids=np.array(kids), kaggle_preds=kpc)
    all_kaggle_preds[exp_name] = {'ids': kids, 'preds': kpc}
    print(f'  done | mean={kpc.mean():.0f} min={kpc.min():.0f} max={kpc.max():.0f}')

    del model; gc.collect()
    if torch.cuda.is_available(): torch.cuda.empty_cache()


convnextv2_base_mm


model.safetensors:   0%|          | 0.00/355M [00:00<?, ?B/s]

  fused=1792
  001/40 | train=0.3886 best=0.3886 | 86s
  002/40 | train=0.1200 best=0.1200 | 65s
  003/40 | train=0.1097 best=0.1097 | 66s
  004/40 | train=0.1079 best=0.1079 | 65s
  005/40 | train=0.1071 best=0.1071 | 67s
  006/40 | train=0.0839 best=0.0839 | 65s
  007/40 | train=0.0909 best=0.0839 | 65s
  008/40 | train=0.0728 best=0.0728 | 66s
  009/40 | train=0.0849 best=0.0728 | 65s
  010/40 | train=0.0783 best=0.0728 | 66s
  011/40 | train=0.0764 best=0.0728 | 65s
  012/40 | train=0.0665 best=0.0665 | 67s
  013/40 | train=0.0603 best=0.0603 | 65s
  014/40 | train=0.0670 best=0.0603 | 67s
  015/40 | train=0.0617 best=0.0603 | 64s
  016/40 | train=0.0539 best=0.0539 | 65s
  017/40 | train=0.0562 best=0.0539 | 65s
  018/40 | train=0.0529 best=0.0529 | 67s
  019/40 | train=0.0547 best=0.0529 | 65s
  020/40 | train=0.0508 best=0.0508 | 67s
  021/40 | train=0.0526 best=0.0508 | 66s
  022/40 | train=0.0469 best=0.0469 | 66s
  023/40 | train=0.0453 best=0.0453 | 67s
  024/40 | train=0.04

model.safetensors:   0%|          | 0.00/218M [00:00<?, ?B/s]

  fused=2048
  001/84 | train=0.5928 best=0.5928 | 106s
  002/84 | train=0.1668 best=0.1668 | 66s
  003/84 | train=0.1612 best=0.1612 | 64s
  004/84 | train=0.1369 best=0.1369 | 67s
  005/84 | train=0.1396 best=0.1369 | 63s
  006/84 | train=0.1173 best=0.1173 | 64s
  007/84 | train=0.1060 best=0.1060 | 64s
  008/84 | train=0.1020 best=0.1020 | 64s
  009/84 | train=0.1016 best=0.1016 | 63s
  010/84 | train=0.0946 best=0.0946 | 65s
  011/84 | train=0.1009 best=0.0946 | 63s
  012/84 | train=0.0958 best=0.0946 | 65s
  013/84 | train=0.0878 best=0.0878 | 63s
  014/84 | train=0.0822 best=0.0822 | 64s
  015/84 | train=0.0870 best=0.0822 | 64s
  016/84 | train=0.0883 best=0.0822 | 63s
  017/84 | train=0.0866 best=0.0822 | 63s
  018/84 | train=0.0800 best=0.0800 | 63s
  019/84 | train=0.0823 best=0.0800 | 62s
  020/84 | train=0.0766 best=0.0766 | 65s
  021/84 | train=0.0765 best=0.0765 | 65s
  022/84 | train=0.0964 best=0.0765 | 63s
  023/84 | train=0.0730 best=0.0730 | 64s
  024/84 | train=0.0

model.safetensors:   0%|          | 0.00/346M [00:00<?, ?B/s]

  fused=1536
  001/42 | train=0.4209 best=0.4209 | 65s
  002/42 | train=0.1477 best=0.1477 | 53s
  003/42 | train=0.1145 best=0.1145 | 54s
  004/42 | train=0.1443 best=0.1145 | 54s
  005/42 | train=0.1183 best=0.1145 | 63s
  006/42 | train=0.0952 best=0.0952 | 65s
  007/42 | train=0.0820 best=0.0820 | 54s
  008/42 | train=0.0866 best=0.0820 | 53s
  009/42 | train=0.0790 best=0.0790 | 64s
  010/42 | train=0.0695 best=0.0695 | 54s
  011/42 | train=0.0674 best=0.0674 | 54s
  012/42 | train=0.0667 best=0.0667 | 53s
  013/42 | train=0.0649 best=0.0649 | 53s
  014/42 | train=0.0546 best=0.0546 | 54s
  015/42 | train=0.0602 best=0.0546 | 54s
  016/42 | train=0.0581 best=0.0546 | 64s
  017/42 | train=0.0581 best=0.0546 | 65s
  018/42 | train=0.0586 best=0.0546 | 66s
  019/42 | train=0.0506 best=0.0506 | 64s
  020/42 | train=0.0481 best=0.0481 | 54s
  021/42 | train=0.0491 best=0.0481 | 54s
  022/42 | train=0.0484 best=0.0481 | 64s
  023/42 | train=0.0469 best=0.0469 | 64s
  024/42 | train=0.04

In [ ]:
exp_name = 'convnextv2_base_augmax'
backbone  = 'convnextv2_base.fcmae_ft_in22k_in1k'
img_size  = 224; bs = 64
EPOCHS    = BEST_EPOCHS.get(exp_name, 50)
ckpt_path  = CKPT_DIR / f'{exp_name}_full.pth'
preds_path = OUT_DIR  / f'{exp_name}_full_preds.npz'
print(f'\n{exp_name} | {EPOCHS} epochs')

if not DEBUG and ckpt_path.exists() and preds_path.exists():
    saved = np.load(preds_path)
    all_kaggle_preds[exp_name] = {'ids': list(saved['kaggle_ids']), 'preds': saved['kaggle_preds']}
    print(f'  skip | {len(saved["kaggle_ids"])} preds')
else:
    gc.collect()
    if torch.cuda.is_available(): torch.cuda.empty_cache()

    class _ImgDS(Dataset):
        def __init__(self, df, tfm, is_train=True):
            self.df = df.reset_index(drop=True); self.tfm = tfm; self.is_train = is_train
        def __len__(self): return len(self.df)
        def __getitem__(self, idx):
            row = self.df.iloc[idx]
            img = cv2.cvtColor(cv2.imread(row.path, cv2.IMREAD_COLOR), cv2.COLOR_BGR2RGB)
            img = self.tfm(image=img)['image']
            return (img, torch.tensor(float(row.target_log), dtype=torch.float32)) if self.is_train else (img, str(row.image_id))

    def _make(df, tfm, is_train, shuffle, drop_last=False):
        return DataLoader(_ImgDS(df, tfm, is_train), batch_size=bs, shuffle=shuffle,
                          num_workers=NUM_WORKERS, pin_memory=True, drop_last=drop_last)

    class _Model(nn.Module):
        def __init__(self):
            super().__init__()
            self.backbone = timm.create_model(backbone, pretrained=True, num_classes=0, global_pool='avg')
            d = self.backbone.num_features
            self.head = nn.Sequential(nn.LayerNorm(d), nn.Linear(d,512), nn.GELU(), nn.Dropout(DROPOUT),
                                      nn.Linear(512,128), nn.GELU(), nn.Dropout(DROPOUT/2), nn.Linear(128,1))
            print(f'  features={d}')
        def forward(self, x): return self.head(self.backbone(x)).squeeze(1)

    train_tfm = A.Compose([
        A.RandomResizedCrop(size=(img_size,img_size), scale=(0.5,1.0), ratio=(0.75,1.33), interpolation=cv2.INTER_AREA),
        A.HorizontalFlip(p=0.5), A.VerticalFlip(p=0.2), A.RandomRotate90(p=0.3),
        A.ColorJitter(brightness=0.4, contrast=0.4, saturation=0.4, hue=0.1, p=0.8),
        A.GaussianBlur(blur_limit=(3,7), p=0.3),
        A.GridDistortion(num_steps=5, distort_limit=0.3, p=0.2),
        A.CoarseDropout(max_holes=8, max_height=32, max_width=32, fill_value=0, p=0.3),
        A.Normalize(mean=(0.485,0.456,0.406), std=(0.229,0.224,0.225)), ToTensorV2(),
    ])
    test_tfm = A.Compose([
        A.LongestMaxSize(max_size=img_size),
        A.PadIfNeeded(min_height=img_size, min_width=img_size, border_mode=cv2.BORDER_REFLECT_101),
        A.CenterCrop(height=img_size, width=img_size),
        A.Normalize(mean=(0.485,0.456,0.406), std=(0.229,0.224,0.225)), ToTensorV2(),
    ])

    tr_l = _make(df_all_train, train_tfm, True,  True,  True)
    ka_l = _make(test_df,      test_tfm,  False, False)

    model     = _Model().to(device)
    bp = [(n,p) for n,p in model.backbone.named_parameters() if p.requires_grad]
    hp = [(n,p) for n,p in model.head.named_parameters()    if p.requires_grad]
    gs = max(1, len(bp)//4); pg = []
    for g in range(4):
        chunk = bp[g*gs:(g+1)*gs if g<3 else len(bp)]
        lr_g  = LR * (0.25**(3-g))
        wd_  = [p for n,p in chunk if p.ndim>1 and 'bias' not in n and 'norm' not in n.lower()]
        nwd  = [p for n,p in chunk if not (p.ndim>1 and 'bias' not in n and 'norm' not in n.lower())]
        if wd_:  pg.append({'params':wd_,  'lr':lr_g, 'weight_decay':WD})
        if nwd:  pg.append({'params':nwd,  'lr':lr_g, 'weight_decay':0.})
    hwd  = [p for n,p in hp if p.ndim>1 and 'bias' not in n and 'norm' not in n.lower()]
    hnwd = [p for n,p in hp if not (p.ndim>1 and 'bias' not in n and 'norm' not in n.lower())]
    if hwd:  pg.append({'params':hwd,  'lr':LR, 'weight_decay':WD})
    if hnwd: pg.append({'params':hnwd, 'lr':LR, 'weight_decay':0.})
    optimizer = torch.optim.AdamW(pg)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS*len(tr_l), eta_min=1e-7)
    scaler    = GradScaler(enabled=torch.cuda.is_available())
    criterion = nn.HuberLoss(delta=0.5)

    best_loss = float('inf')
    for epoch in range(1, EPOCHS+1):
        t0 = time.time(); model.train(); total = 0.; optimizer.zero_grad(set_to_none=True)
        for imgs, targets in tr_l:
            imgs = imgs.to(device, non_blocking=True); targets = targets.to(device, non_blocking=True)
            if random.random() < AUG_MIX_PROB:
                lam = np.random.beta(MIXUP_ALPHA, MIXUP_ALPHA); idx = torch.randperm(imgs.size(0), device=imgs.device)
                imgs = lam*imgs+(1-lam)*imgs[idx]; targets = lam*targets+(1-lam)*targets[idx]
            with autocast(device_type='cuda', enabled=torch.cuda.is_available()):
                loss = criterion(model(imgs), targets)
            scaler.scale(loss).backward(); scaler.unscale_(optimizer)
            nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            scaler.step(optimizer); scaler.update(); optimizer.zero_grad(set_to_none=True)
            if scheduler: scheduler.step()
            total += float(loss.item())
        train_loss = total / len(tr_l)
        if train_loss < best_loss:
            best_loss = train_loss
            torch.save({'model_state': model.state_dict(), 'epoch': epoch,
                        'backbone': backbone, 'img_size': img_size}, ckpt_path)
        print(f'  {epoch:03d}/{EPOCHS} | train={train_loss:.4f} best={best_loss:.4f} | {time.time()-t0:.0f}s', flush=True)

    ckpt = torch.load(ckpt_path, map_location=device, weights_only=False)
    model.load_state_dict(ckpt['model_state']); model.eval()

    all_ids2 = []; all_p2 = []
    with torch.no_grad():
        for imgs, ids in ka_l:
            imgs = imgs.to(device, non_blocking=True)
            with autocast(device_type='cuda', enabled=torch.cuda.is_available()):
                out = model(imgs); out_flip = model(torch.flip(imgs, dims=[3]))
            all_p2.append(((out+out_flip)/2).float().cpu().numpy())
            all_ids2.extend(ids if isinstance(ids, list) else list(ids))
    kpl = np.concatenate(all_p2)
    kpc = preds_to_calories(kpl)
    np.savez(preds_path, kaggle_ids=np.array(all_ids2), kaggle_preds=kpc)
    all_kaggle_preds[exp_name] = {'ids': all_ids2, 'preds': kpc}
    print(f'  done | mean={kpc.mean():.0f} min={kpc.min():.0f} max={kpc.max():.0f}')
    del model; gc.collect()
    if torch.cuda.is_available(): torch.cuda.empty_cache()


convnextv2_base_augmax | 79 epochs
  features=1024
  001/79 | train=0.4822 best=0.4822 | 59s
  002/79 | train=0.1330 best=0.1330 | 60s
  003/79 | train=0.1176 best=0.1176 | 61s
  004/79 | train=0.1032 best=0.1032 | 61s
  005/79 | train=0.0947 best=0.0947 | 62s
  006/79 | train=0.0968 best=0.0947 | 60s
  007/79 | train=0.0878 best=0.0878 | 62s
  008/79 | train=0.0833 best=0.0833 | 60s
  009/79 | train=0.0841 best=0.0833 | 60s
  010/79 | train=0.0781 best=0.0781 | 61s
  011/79 | train=0.0743 best=0.0743 | 61s
  012/79 | train=0.0747 best=0.0743 | 59s
  013/79 | train=0.0702 best=0.0702 | 62s
  014/79 | train=0.0690 best=0.0690 | 61s
  015/79 | train=0.0688 best=0.0688 | 61s
  016/79 | train=0.0675 best=0.0675 | 59s
  017/79 | train=0.0663 best=0.0663 | 61s
  018/79 | train=0.0709 best=0.0663 | 60s
  019/79 | train=0.0645 best=0.0645 | 60s
  020/79 | train=0.0651 best=0.0645 | 59s
  021/79 | train=0.0660 best=0.0645 | 60s
  022/79 | train=0.0635 best=0.0635 | 60s
  023/79 | train=0.0620 

In [ ]:
# Swin-Base MM
exp_name      = 'swin_base_mm'
backbone_name = 'swin_base_patch4_window7_224.ms_in22k_ft_in1k'
img_size      = 224; bs = 32; EPOCHS_SWIN = 40
ckpt_path  = CKPT_DIR / f'{exp_name}_full.pth'
preds_path = OUT_DIR  / f'{exp_name}_full_preds.npz'
print(f'\n{exp_name} | {EPOCHS_SWIN} epochs')

if ckpt_path.exists() and preds_path.exists():
    saved = np.load(preds_path)
    all_kaggle_preds[exp_name] = {'ids': list(saved['kaggle_ids']), 'preds': saved['kaggle_preds']}
    print(f'  skip | {len(saved["kaggle_ids"])} preds')
else:
    gc.collect()
    if torch.cuda.is_available(): torch.cuda.empty_cache()

    train_tfm = A.Compose([
        A.RandomResizedCrop(size=(img_size,img_size), scale=(0.65,1.0), ratio=(0.80,1.20), interpolation=cv2.INTER_AREA),
        A.HorizontalFlip(p=0.5),
        A.ColorJitter(brightness=0.25, contrast=0.25, saturation=0.25, hue=0.08, p=0.6),
        A.Normalize(mean=(0.485,0.456,0.406), std=(0.229,0.224,0.225)), ToTensorV2(),
    ])
    test_tfm = A.Compose([
        A.LongestMaxSize(max_size=img_size),
        A.PadIfNeeded(min_height=img_size, min_width=img_size, border_mode=cv2.BORDER_REFLECT_101),
        A.CenterCrop(height=img_size, width=img_size),
        A.Normalize(mean=(0.485,0.456,0.406), std=(0.229,0.224,0.225)), ToTensorV2(),
    ])

    train_loader  = make_loader(df_all_train, train_tfm, bs, is_train=True,  shuffle=True, drop_last=True)
    kaggle_loader = make_loader(test_df,      test_tfm,  bs, is_train=False, shuffle=False)

    model     = MultimodalRegressor(backbone_name, img_size, TEXT_DIM, DROPOUT).to(device)
    optimizer = get_optimizer_llrd(model, base_lr=LR, wd=WD)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS_SWIN*len(train_loader), eta_min=1e-7)
    scaler    = GradScaler(enabled=torch.cuda.is_available())
    criterion = nn.HuberLoss(delta=0.5)

    best_loss = float('inf')
    for epoch in range(1, EPOCHS_SWIN+1):
        t0         = time.time()
        train_loss = train_one_epoch(model, train_loader, optimizer, scheduler, scaler, criterion)
        if train_loss < best_loss:
            best_loss = train_loss
            torch.save({'model_state': model.state_dict(), 'epoch': epoch,
                        'backbone': backbone_name, 'img_size': img_size}, ckpt_path)
        print(f'  {epoch:03d}/{EPOCHS_SWIN} | train={train_loss:.4f} best={best_loss:.4f} | {time.time()-t0:.0f}s', flush=True)

    ckpt = torch.load(ckpt_path, map_location=device, weights_only=False)
    model.load_state_dict(ckpt['model_state'])
    kids, kpl = predict(model, kaggle_loader)
    kpc = preds_to_calories(kpl)
    np.savez(preds_path, kaggle_ids=np.array(kids), kaggle_preds=kpc)
    all_kaggle_preds[exp_name] = {'ids': kids, 'preds': kpc}
    print(f'  done | mean={kpc.mean():.0f} min={kpc.min():.0f} max={kpc.max():.0f}')
    del model; gc.collect()
    if torch.cuda.is_available(): torch.cuda.empty_cache()


swin_base_mm | 40 epochs


model.safetensors:   0%|          | 0.00/353M [00:00<?, ?B/s]

  fused=1792
  001/40 | train=0.2776 best=0.2776 | 64s
  002/40 | train=0.1458 best=0.1458 | 64s
  003/40 | train=0.1337 best=0.1337 | 66s
  004/40 | train=0.1050 best=0.1050 | 66s
  005/40 | train=0.0942 best=0.0942 | 65s
  006/40 | train=0.0993 best=0.0942 | 65s
  007/40 | train=0.0902 best=0.0902 | 64s
  008/40 | train=0.0839 best=0.0839 | 66s
  009/40 | train=0.0854 best=0.0839 | 65s
  010/40 | train=0.0833 best=0.0833 | 65s
  011/40 | train=0.0791 best=0.0791 | 64s
  012/40 | train=0.0688 best=0.0688 | 66s
  013/40 | train=0.0699 best=0.0688 | 65s
  014/40 | train=0.0704 best=0.0688 | 64s
  015/40 | train=0.0655 best=0.0655 | 67s
  016/40 | train=0.0609 best=0.0609 | 66s
  017/40 | train=0.0577 best=0.0577 | 66s
  018/40 | train=0.0575 best=0.0575 | 67s
  019/40 | train=0.0551 best=0.0551 | 65s
  020/40 | train=0.0521 best=0.0521 | 67s
  021/40 | train=0.0516 best=0.0516 | 65s
  022/40 | train=0.0496 best=0.0496 | 65s
  023/40 | train=0.0500 best=0.0496 | 64s
  024/40 | train=0.04

In [ ]:
# Swin-Base MM
exp_name      = 'swin_base_mm'
backbone_name = 'swin_base_patch4_window7_224.ms_in22k_ft_in1k'
img_size      = 224; bs = 32; EPOCHS_SWIN = 40
ckpt_path  = CKPT_DIR / f'{exp_name}_full.pth'
preds_path = OUT_DIR  / f'{exp_name}_full_preds.npz'
print(f'\n{exp_name} | {EPOCHS_SWIN} epochs')

if ckpt_path.exists() and preds_path.exists():
    saved = np.load(preds_path)
    all_kaggle_preds[exp_name] = {'ids': list(saved['kaggle_ids']), 'preds': saved['kaggle_preds']}
    print(f'  skip | {len(saved["kaggle_ids"])} preds')
else:
    gc.collect()
    if torch.cuda.is_available(): torch.cuda.empty_cache()

    train_tfm = A.Compose([
        A.RandomResizedCrop(size=(img_size,img_size), scale=(0.65,1.0), ratio=(0.80,1.20), interpolation=cv2.INTER_AREA),
        A.HorizontalFlip(p=0.5),
        A.ColorJitter(brightness=0.25, contrast=0.25, saturation=0.25, hue=0.08, p=0.6),
        A.Normalize(mean=(0.485,0.456,0.406), std=(0.229,0.224,0.225)), ToTensorV2(),
    ])
    test_tfm = A.Compose([
        A.LongestMaxSize(max_size=img_size),
        A.PadIfNeeded(min_height=img_size, min_width=img_size, border_mode=cv2.BORDER_REFLECT_101),
        A.CenterCrop(height=img_size, width=img_size),
        A.Normalize(mean=(0.485,0.456,0.406), std=(0.229,0.224,0.225)), ToTensorV2(),
    ])

    train_loader  = make_loader(df_all_train, train_tfm, bs, is_train=True,  shuffle=True, drop_last=True)
    kaggle_loader = make_loader(test_df,      test_tfm,  bs, is_train=False, shuffle=False)

    model     = MultimodalRegressor(backbone_name, img_size, TEXT_DIM, DROPOUT).to(device)
    optimizer = get_optimizer_llrd(model, base_lr=LR, wd=WD)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS_SWIN*len(train_loader), eta_min=1e-7)
    scaler    = GradScaler(enabled=torch.cuda.is_available())
    criterion = nn.HuberLoss(delta=0.5)

    best_loss = float('inf')
    for epoch in range(1, EPOCHS_SWIN+1):
        t0         = time.time()
        train_loss = train_one_epoch(model, train_loader, optimizer, scheduler, scaler, criterion)
        if train_loss < best_loss:
            best_loss = train_loss
            torch.save({'model_state': model.state_dict(), 'epoch': epoch,
                        'backbone': backbone_name, 'img_size': img_size}, ckpt_path)
        print(f'  {epoch:03d}/{EPOCHS_SWIN} | train={train_loss:.4f} best={best_loss:.4f} | {time.time()-t0:.0f}s', flush=True)

    ckpt = torch.load(ckpt_path, map_location=device, weights_only=False)
    model.load_state_dict(ckpt['model_state'])
    kids, kpl = predict(model, kaggle_loader)
    kpc = preds_to_calories(kpl)
    np.savez(preds_path, kaggle_ids=np.array(kids), kaggle_preds=kpc)
    all_kaggle_preds[exp_name] = {'ids': kids, 'preds': kpc}
    print(f'  done | mean={kpc.mean():.0f} min={kpc.min():.0f} max={kpc.max():.0f}')
    del model; gc.collect()
    if torch.cuda.is_available(): torch.cuda.empty_cache()


swin_base_mm | 40 epochs
  skip | 547 preds


In [ ]:
# Blend only BLEND_MODELS (excludes efficientnet whose NM weight was 0)
BLEND_MODELS = ['convnextv2_base_mm', 'deit3_base_mm', 'convnextv2_base_augmax', 'swin_base_mm', 'coatnet1_imgonly']
BLEND_FALLBACK_W = {'convnextv2_base_mm': 0.40, 'deit3_base_mm': 0.30, 'convnextv2_base_augmax': 0.10,
                    'swin_base_mm': 0.15, 'coatnet1_imgonly': 0.05}

available = [n for n in BLEND_MODELS if n in all_kaggle_preds]
kaggle_ids = all_kaggle_preds[available[0]]['ids']
kag_preds  = np.stack([all_kaggle_preds[n]['preds'] for n in available])
M = len(available)

if NM_WEIGHTS is not None:
    w = np.array([NM_WEIGHTS.get(n, BLEND_FALLBACK_W.get(n, 1.0/M)) for n in available])
else:
    w = np.array([BLEND_FALLBACK_W.get(n, 1.0/M) for n in available])

w = np.abs(w); w /= w.sum()

blend = (kag_preds * w[:, None]).sum(axis=0)
blend = np.clip(blend, PRED_MIN, PRED_MAX)

for n, wi in zip(available, w):
    print(f'  {n}: {wi:.4f}')
print(f'blend: mean={blend.mean():.0f} min={blend.min():.0f} max={blend.max():.0f}')

  convnextv2_base_mm: 0.0591
  deit3_base_mm: 0.0918
  convnextv2_base_augmax: 0.7187
  swin_base_mm: 0.1304
blend: mean=313 min=59 max=2814


In [ ]:
# Blend only BLEND_MODELS (excludes efficientnet whose NM weight was 0)
available = [n for n in BLEND_MODELS if n in all_kaggle_preds]
kaggle_ids = all_kaggle_preds[available[0]]['ids']
kag_preds  = np.stack([all_kaggle_preds[n]['preds'] for n in available])
M = len(available)

# Use NM weights from step 1 when available, otherwise fall back to BLEND_FALLBACK_W
if NM_WEIGHTS is not None:
    w = np.array([NM_WEIGHTS.get(n, BLEND_FALLBACK_W.get(n, 1.0/M)) for n in available])
else:
    w = np.array([BLEND_FALLBACK_W.get(n, 1.0/M) for n in available])

w = np.abs(w); w /= w.sum()

blend = (kag_preds * w[:, None]).sum(axis=0)
blend = np.clip(blend, PRED_MIN, PRED_MAX)

for n, wi in zip(available, w):
    print(f'  {n}: {wi:.4f}')
print(f'blend: mean={blend.mean():.0f} min={blend.min():.0f} max={blend.max():.0f}')

  convnextv2_base_mm: 0.0591
  deit3_base_mm: 0.0918
  convnextv2_base_augmax: 0.7187
  swin_base_mm: 0.1304
blend: mean=313 min=59 max=2814


In [ ]:
sub = pd.DataFrame({'image_id': kaggle_ids, 'calories': blend.round(2)})
sub_path = WORK / 'submission_final.csv'
sub.to_csv(sub_path, index=False)
print(sub_path)
print(sub.head())

/content/drive/MyDrive/food_step2/submission_final.csv
    image_id  calories
0  test_0000     90.69
1  test_0001    141.88
2  test_0002    105.77
3  test_0003    231.31
4  test_0004    174.34


In [ ]:
import subprocess
result = subprocess.run(
    ['kaggle', 'competitions', 'submit',
     '-c', 'm2-food-calorie-estimation',
     '-f', str(sub_path),
     '-m', f'step2 full train blend w={w.round(3).tolist()}'],
    capture_output=True, text=True
)
print(result.stdout or result.stderr)

Successfully submitted to M2 IASD Project - Food Calorie Estimation
